# DE1 — Final Project Notebook
>**Students:** Maxence DELEHELLE, Amine SAAD-EDDINE   
>**Teacher:** Badr TAJINI     
>**Academic year:** 2025–2026  
>**Program:** Data & Applications - Engineering - (FD)   
>**Course:** Data Engineering I  
---

## 0. Load config

In [11]:
import yaml, pathlib, datetime
from pyspark.sql import SparkSession, functions as F, types as T

with open("project/de1_project_config.yml") as f:
    CFG = yaml.safe_load(f)

spark = SparkSession.builder.appName("de1-project").getOrCreate()
CFG


{'paths': {'raw_csv_glob': 'data/parking_violation.csv',
  'bronze': 'outputs/project/bronze/',
  'silver': 'outputs/project/silver/',
  'gold': 'outputs/project/gold/',
  'proof': 'proof/'},
 'layout': {'partition_by': ['date']},
 'slos': {'freshness_hours': 2,
  'query_latency_q1_seconds': 4,
  'storage_reduction_ratio': 0.6},
 'schema': {'enforce_types': True, 'drop_nulls': True}}

In [ ]:
import requests
import pandas as pd
import os
from datetime import datetime

def export_spark_metrics(spark, run_id, note="", output_csv="lab1_metrics_log.csv"):
    """
    Exporte les métriques Spark (stages) dans un CSV.
    - spark: SparkSession active
    - run_id: identifiant de ton exécution (ex: "r1", "baseline", etc.)
    - note: commentaire ou variante testée (ex: "broadcast join", etc.)
    - output_csv: nom du fichier de sortie
    """
    ui_url = spark.sparkContext.uiWebUrl
    app_id = spark.sparkContext.applicationId
    
    if not ui_url:
        print("⚠️ Pas d'UI Spark détectée (aucun job en cours ?)")
        return
    
    try:
        # Récupérer les infos de stages
        stages_url = f"{ui_url}/api/v1/applications/{app_id}/stages"
        stages = requests.get(stages_url).json()
    except Exception as e:
        print(f"❌ Erreur API Spark: {e}")
        return
    
    data = []
    for s in stages:
        stage_id = s.get("stageId")
        job_ids = s.get("jobIds", [])
        metrics = s.get("executorSummary", {})
        
        # Métriques de base
        input_bytes = s.get("inputBytes", 0)
        shuffle_read = s.get("shuffleReadBytes", 0)
        shuffle_write = s.get("shuffleWriteBytes", 0)
        num_tasks = s.get("numTasks", 0)
        completion_time = s.get("completionTime", None)
        
        # Fallback pour Spark < 3.x
        if not completion_time and "completionTime" in s.get("completionTime", {}):
            completion_time = s["completionTime"]
        
        data.append({
            "run_id": run_id,
            "job_id": job_ids[0] if job_ids else None,
            "stage_id": stage_id,
            "task": s.get("name", ""),
            "note": note,
            "files_read": s.get("inputRecords", 0),
            "input_size_bytes": input_bytes,
            "shuffle_read_bytes": shuffle_read,
            "shuffle_write_bytes": shuffle_write,
            "timestamp": datetime.now().isoformat()
        })
    
    df = pd.DataFrame(data)
    
    if df.empty:
        print("⚠️ Aucune métrique trouvée (aucun stage terminé ?)")
        return

    # Écrire ou ajouter au CSV
    if os.path.exists(output_csv):
        df.to_csv(output_csv, mode="a", header=False, index=False)
    else:
        df.to_csv(output_csv, index=False)

    print(f"✅ {len(df)} lignes exportées vers {output_csv}")

## 1. Bronze — landing raw data

In [12]:
raw_glob = CFG["paths"]["raw_csv_glob"]
bronze = CFG["paths"]["bronze"]
proof = CFG["paths"]["proof"]

df_raw = (spark.read.option("header","true").csv(raw_glob))
df_raw.write.mode("overwrite").csv(bronze)  # keep raw as CSV copy
print("Bronze written:", bronze)


26/01/02 10:53:06 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Summons_Number, Plate_ID, Registration_State, Plate_Type, Issue_Date, Violation_Code, Vehicle_Body_Type, Vehicle_Make, Issuing_Agency, Street_Code1, Street_Code2, Street_Code3, Vehicle_Expiration_Date, Violation_Location, Violation_Precinct, Issuer_Precinct, Issuer_Code, Issuer_Command, Issuer_Squad, Violation_Time, Time_First_Observed, Violation_County, Violation_In_Front_Of_Or_Opposite, House_Number, Street_Name, Intersecting_Street, Date_First_Observed, Law_Section, Sub_Division, Violation_Legal_Code, Days_Parking_In_Effect____, From_Hours_In_Effect, To_Hours_In_Effect, Vehicle_Color, Unregistered_Vehicle?, Vehicle_Year, Meter_Number, Feet_From_Curb, Violation_Post_Code, Violation_Description, No_Standing_or_Stopping_Violation, Hydrant_Violation, Double_Parking_Violation, Latitude, Longitude, Community_Board, Community_Council_, Census_Tract, BIN, BBL, NTA, year, month
 Schema: _c0, Summon

Bronze written: outputs/project/bronze/


In [13]:
df_raw.printSchema()

root
 |-- _c0: string (nullable = true)
 |-- Summons_Number: string (nullable = true)
 |-- Plate_ID: string (nullable = true)
 |-- Registration_State: string (nullable = true)
 |-- Plate_Type: string (nullable = true)
 |-- Issue_Date: string (nullable = true)
 |-- Violation_Code: string (nullable = true)
 |-- Vehicle_Body_Type: string (nullable = true)
 |-- Vehicle_Make: string (nullable = true)
 |-- Issuing_Agency: string (nullable = true)
 |-- Street_Code1: string (nullable = true)
 |-- Street_Code2: string (nullable = true)
 |-- Street_Code3: string (nullable = true)
 |-- Vehicle_Expiration_Date: string (nullable = true)
 |-- Violation_Location: string (nullable = true)
 |-- Violation_Precinct: string (nullable = true)
 |-- Issuer_Precinct: string (nullable = true)
 |-- Issuer_Code: string (nullable = true)
 |-- Issuer_Command: string (nullable = true)
 |-- Issuer_Squad: string (nullable = true)
 |-- Violation_Time: string (nullable = true)
 |-- Time_First_Observed: string (nullable

## 2. Silver — cleaning and typing

In [14]:
silver = CFG["paths"]["silver"]

# Example typing; adapt to dataset
from pyspark.sql import functions as F, types as T

df_silver = (df_raw
    .withColumn("metric", F.col("Violation_Code").cast("double"))
    .withColumn("date", F.to_date(F.col("Issue_Date"), "MM/dd/yyyy"))
    .dropna(subset=["metric", "date"]))

df_silver.write.mode("overwrite").parquet(silver)

print("Silver written:", silver)


26/01/02 10:54:57 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Summons_Number, Plate_ID, Registration_State, Plate_Type, Issue_Date, Violation_Code, Vehicle_Body_Type, Vehicle_Make, Issuing_Agency, Street_Code1, Street_Code2, Street_Code3, Vehicle_Expiration_Date, Violation_Location, Violation_Precinct, Issuer_Precinct, Issuer_Code, Issuer_Command, Issuer_Squad, Violation_Time, Time_First_Observed, Violation_County, Violation_In_Front_Of_Or_Opposite, House_Number, Street_Name, Intersecting_Street, Date_First_Observed, Law_Section, Sub_Division, Violation_Legal_Code, Days_Parking_In_Effect____, From_Hours_In_Effect, To_Hours_In_Effect, Vehicle_Color, Unregistered_Vehicle?, Vehicle_Year, Meter_Number, Feet_From_Curb, Violation_Post_Code, Violation_Description, No_Standing_or_Stopping_Violation, Hydrant_Violation, Double_Parking_Violation, Latitude, Longitude, Community_Board, Community_Council_, Census_Tract, BIN, BBL, NTA, year, month
 Schema: _c0, Summon

Silver written: outputs/project/silver/


## 3. Gold — analytics tables

In [15]:
gold = CFG["paths"]["gold"]
partition_by = CFG["layout"]["partition_by"]

# Example gold Q1
gold_q1 = (df_silver.groupBy("date").agg(F.sum("metric").alias("sum_metric")))
(gold_q1.write.mode("overwrite").partitionBy(*partition_by).parquet(f"{gold}/q1_daily"))

# Gold Q2: Violations by Location (Precinct/County)
gold_q2 = (df_silver
    .groupBy("Violation_Precinct", "Violation_County")
    .agg(F.count("metric").alias("ticket_count"),
         F.sum("metric").alias("total_revenue"))
)
gold_q2.write.mode("overwrite").parquet(f"{gold}/q2_geographic")

# Gold Q3: Performance by Vehicle Year and Month
gold_q3 = (df_silver
    .groupBy("year", "month", "Vehicle_Make")
    .agg(F.count("metric").alias("violations_by_make"))
)
gold_q3.write.mode("overwrite").parquet(f"{gold}/q3_vehicle_trends")

print("Gold written:", gold)


Gold written: outputs/project/gold/


## 4. Baseline plans and metrics

In [16]:
import os, datetime as _dt, pathlib
pathlib.Path(proof).mkdir(parents=True, exist_ok=True)

# Example baseline plan
plan1 = gold_q1._jdf.queryExecution().executedPlan().toString()
with open(f"{proof}/baseline_q1_plan.txt","w") as f:
    f.write(str(_dt.datetime.now())+"\n")
    f.write(plan1)
    
plan2 = gold_q2._jdf.queryExecution().executedPlan().toString()
with open(f"{proof}/baseline_q2_plan.txt","w") as f:
    f.write(str(_dt.datetime.now())+"\n")
    f.write(plan2)

plan3 = gold_q3._jdf.queryExecution().executedPlan().toString()
with open(f"{proof}/baseline_q3_plan.txt","w") as f:
    f.write(str(_dt.datetime.now())+"\n")
    f.write(plan3)
    
print("Saved baseline plan. Record Spark UI metrics now.")


Saved baseline plan. Record Spark UI metrics now.


## 5. Optimization — layout and joins

In [17]:
# Example: narrow projection and pre‑aggregation before write
df_silver_min = df_silver.select("date","metric")
gold_q1_opt = (df_silver_min.groupBy("date").agg(F.sum("metric").alias("sum_metric")))
gold_q1_opt.write.mode("overwrite").partitionBy(*partition_by).parquet(f"{gold}/q1_daily_opt")

plan_opt = gold_q1_opt._jdf.queryExecution().executedPlan().toString()
with open(f"{proof}/optimized_q1_plan.txt","w") as f:
    f.write(str(_dt.datetime.now())+"\n")
    f.write(plan_opt)
print("Saved optimized plan. Record Spark UI metrics now.")


Saved optimized plan. Record Spark UI metrics now.


## 6. Cleanup

In [18]:
spark.stop()
print("Spark session stopped.")


Spark session stopped.
